In [ ]:
%logstart -o notebook_log.txt append

In [ ]:
import sys
from pathlib import Path
print(Path.cwd())
import chromadb
from vorstellungsgesprach.notebook import prepare_jobs
from vorstellungsgesprach.key_terms import analyze_jobs
from vorstellungsgesprach.chunck import build_documents
from vorstellungsgesprach.utils import load_data, add_language_metadata,remove_duplicate_jobs
from vorstellungsgesprach import conf
from vorstellungsgesprach import embeddings
from vorstellungsgesprach import evaluation
from vorstellungsgesprach import conf, rag
from vorstellungsgesprach.models import list_available_chat_models
from google import genai
import json
import os




In [ ]:
import vorstellungsgesprach.chunck as chunck_module
print(dir(chunck_module))

In [ ]:
#defining API key for GenAI
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [ ]:
jobs = prepare_jobs("../data/raw/job.json")
print(len(jobs))

In [ ]:
sample_jobs = jobs[:5]
analyzed_jobs = analyze_jobs(sample_jobs)


In [ ]:
print(analyzed_jobs[0].keys())
print(json.dumps(analyzed_jobs[0]["competencies"], ensure_ascii=False, indent=2))

In [ ]:
with open("../data/processed/key_terms_sample.json", "w", encoding="utf-8") as f:
    json.dump(analyzed_jobs, f, ensure_ascii=False, indent=2)

In [ ]:
from collections import defaultdict

def build_keyword_index(extractions):
    keyword_to_jobs = defaultdict(list)
    keyword_to_contexts = defaultdict(set)

    for entry in extractions:
        for comp in entry["competencies"]:
            term_key = comp["canonical_name"].strip().lower()
            if not term_key:
                continue

            keyword_to_jobs[term_key].append({
                "title": entry.get("title"),
                "company": entry.get("company"),
            })
            keyword_to_contexts[term_key].add(comp["context"])

    return {
        "keyword_to_jobs": dict(keyword_to_jobs),
        "keyword_to_contexts": {k: sorted(v) for k, v in keyword_to_contexts.items()},
    }

keyword_index = build_keyword_index(analyzed_jobs)

with open("../data/processed/keyword_index_sample.json", "w", encoding="utf-8") as f:
    json.dump(keyword_index, f, ensure_ascii=False, indent=2)

print(f"Keywords únicas nas 5 vagas: {len(keyword_index['keyword_to_jobs'])}")
print(list(keyword_index["keyword_to_jobs"].keys()))

In [ ]:
test_keyword = "python"

print("=== VAGAS ===")
for job in keyword_index["keyword_to_jobs"].get(test_keyword, []):
    print(f"- {job['title']} @ {job['company']}")

print("\n=== CONTEXTOS ===")
for ctx in keyword_index["keyword_to_contexts"].get(test_keyword, []):
    print(f"- {ctx}\n")

In [ ]:
documents = build_documents(jobs)
print(f"Total de vagas: {len(jobs)}")
print(f"Total de documentos (após chunking): {len(documents)}")

In [ ]:
#create documents for embedding
texts = [doc["text"] for doc in documents]
vectors = embeddings.embed_texts(texts)

print(f"Embeddings gerados: {len(vectors)}")
print(f"Dimensão de cada vetor: {len(vectors[0])}")

In [ ]:
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="vagas_ti")

collection.add(
    ids=[doc["id"] for doc in documents],
    embeddings=vectors,
    documents=[doc["text"] for doc in documents],
    metadatas=[doc["metadata"] for doc in documents],
)

print(f"Documentos indexados: {collection.count()}")

In [ ]:
#check models available
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "vagas_ti"


chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

question = "Welche Stellen erfordern Erfahrung mit Python?"

gemini_client = genai.Client(api_key=conf.GEMINI_API_KEY)
available_models = list_available_chat_models(gemini_client)

results = []

for model_name in available_models:
    try:
        result = rag.answer(
            collection=collection,
            query=question,
            model=model_name,
            n_results=20,
        )

        results.append(result)

        print(f"\n{'=' * 80}")
        print(f"Modelo: {result['model']}")
        print(f"{'=' * 80}")
        print(result["answer"])

    except Exception:
        continue

if results:
    print("\nFontes recuperadas:")

    for index, source in enumerate(results[0]["sources"], start=1):
        metadata = source["metadata"]

        print(
            f"{index}. {metadata.get('title', 'Sem título')} "
            f"@ {metadata.get('company', 'Sem empresa')} "
            f"(distância: {source['distance']:.3f})"
        )
else:
    print("Nenhum modelo produziu uma resposta.")

In [ ]:
judge_model = available_models[0]
judged_results = []

for result in results:
    context = "\n\n".join(
        source["text"]
        for source in result["sources"]
    )

    try:
        scores = evaluation.judge_answer(
            query=result["query"],
            answer=result["answer"],
            context=context,
            judge_model=judge_model,
        )

        judged_results.append(
            {
                **result,
                **scores,
            }
        )

    except Exception:
        continue


best = sorted(
    judged_results,
    key=lambda result: result["overall"] or 0,
    reverse=True,
)


for result in best:
    print(
        f"{result['model']} | "
        f"faithfulness={result['faithfulness']} | "
        f"helpfulness={result['helpfulness']} | "
        f"overall={result['overall']}"
    )


winner = best[0]

print(
    f"\nMelhor modelo: {winner['model']} "
    f"(nota: {winner['overall']})"
)